Undersampling + Residual BLock + Sliding Window CNN + LSTM (no Attention)

In [3]:
import pandas as pd
train_df = pd.read_csv(r"train.csv")
valid_df = pd.read_csv(r"val.csv")
test_df = pd.read_csv(r"test.csv")

train_df.sort_values(by='timestamp', inplace=True)
valid_df.sort_values(by='timestamp', inplace=True)
test_df.sort_values(by='timestamp', inplace=True)

cols_to_drop = ['timestamp', 'network_ips_dst', 'network_ips_src', 'network_ports_dst', 'network_ports_src']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [4]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # tạo mảng index cho step = 1 (dành riêng cho các class hiếm cần bảo toàn)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        # tạo mảng index theo step được truyền vào (cho các class còn lại)
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        
        # lặp qua từng class (Lấy từ step=1 để đảm bảo không sót class nào)
        classes = np.unique(window_labels_step1)
        rng = np.random.default_rng(seed=42)
        for c in classes:
            # ƯU TIÊN GIỮ NGUYÊN băm cửa sổ 1-1 cho Class 2 và 6
            if c in [100]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
                
            count = len(c_indices)
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                 
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [2, 6]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt tại index thực sự
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 1 
NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1) 
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, set Train Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giảm từ 95760 xuống 20000 cửa sổ.
Class 1: Giữ nguyên 1307 cửa sổ (step=1).
Class 2: Giữ nguyên 12639 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giữ nguyên 12894 cửa sổ (step=1).
Class 4: Giữ nguyên 5278 cửa sổ (step=1).
Class 5: Giữ nguyên 5643 cửa sổ (step=1).
Class 6: Giảm từ 23544 xuống 20000 cửa sổ.
Class 7: Giữ nguyên 1957 cửa sổ (step=1).
Tổng số cửa sổ sau khi lọc và Undersampling: 79718
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 20520 cửa sổ (step=1).
Class 1: Giữ nguyên 280 cửa sổ (step=1).
Class 2: Giữ nguyên 2708 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giữ nguyên 2763 cửa sổ (step=1).
Class 4: Giữ nguyên 1131 cửa sổ (step=1).
Class 5: Giữ nguyên 1209 cửa sổ (step=1).
Class 6: Giữ nguyên 5038 cửa sổ (

In [6]:
import torch
import torch.nn as nn

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# MODEL CNN-BiLSTM (ĐÃ LƯỢC BỎ ATTENTION)
class CNN_BiLSTM_NoAttention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_NoAttention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        # Đã loại bỏ self.attention
        
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) 
        
        # Đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # CHUẨN HOÁ LAYER NORM THEO TỪNG TIME-STEP
        out = self.layer_norm(out)
        
        # THAY THẾ ATTENTION: Sử dụng Global Average Pooling theo chiều sequence
        # Lấy trung bình tất cả các time-steps để nén về (Batch, hidden_size * 2)
        context_vector = torch.mean(out, dim=1)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

In [7]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
# tạo folder để lưu model
import os
model_dir = "model_final"
os.makedirs(model_dir, exist_ok=True)

model = CNN_BiLSTM_NoAttention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val macro f1 không cải thiện sau 2 epochs (lưu ý đổi mode thành 'max')
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    # nếu là epoch chẵn thì không đánh giá trên tập val để tiết kiệm thời gian, chỉ đánh giá trên tập val ở các epoch lẻ

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Đưa val_macro_f1 vào scheduler (trước đó là avg_val_loss)
    scheduler.step(val_macro_f1)
    
    # Lưu model dựa trên Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_v2.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Macro F1 sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, zero_division=0, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 0.9101, Train Macro F1: 0.7519 | Val Loss: 0.6746, Val Macro F1: 0.7309


Epoch [2/60] - Train Loss: 0.6120, Train Macro F1: 0.9428 | Val Loss: 0.8395, Val Macro F1: 0.6576


Epoch [3/60] - Train Loss: 0.5838, Train Macro F1: 0.9606 | Val Loss: 0.7825, Val Macro F1: 0.6849


Epoch [4/60] - Train Loss: 0.5700, Train Macro F1: 0.9662 | Val Loss: 0.7017, Val Macro F1: 0.7735


...

Epoch [24/60] - Train Loss: 0.5241, Train Macro F1: 0.9871 | Val Loss: 0.7254, Val Macro F1: 0.7691


Epoch [25/60] - Train Loss: 0.5244, Train Macro F1: 0.9869 | Val Loss: 0.7358, Val Macro F1: 0.7505


Epoch [26/60] - Train Loss: 0.5240, Train Macro F1: 0.9877 | Val Loss: 0.7271, Val Macro F1: 0.7678


Epoch [27/60] - Train Loss: 0.5238, Train Macro F1: 0.9861 | Val Loss: 0.7301, Val Macro F1: 0.7581

[Early Stopping] Model không cải thiện Macro F1 sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.9776    0.9962    0.9868     20520
           1     0.0804    0.1530    0.1054       281
           2     0.6646    0.3226    0.4344      2709
           3     0.6811    0.6431    0.6616      2763
           4     0.8066    0.9982    0.8922      1132
           5     0.9763    0.3058    0.4657      1210
           6     0.7929    1.0000    0.8845      5039
           7     0.7105    0.9643    0.8182       420

    accuracy                         0.8828     34074
   macro avg     0.7113    0.6729    0.6561     34074
weighted avg     0.8850    0.8828    0.8704     34074



UnderSampling + Residual Block + CNN + LSTM + Attention (no Sliding Window)

In [8]:
import pandas as pd
train_df = pd.read_csv(r"train.csv")
valid_df = pd.read_csv(r"val.csv")
test_df = pd.read_csv(r"test.csv")

train_df.sort_values(by='timestamp', inplace=True)
valid_df.sort_values(by='timestamp', inplace=True)
test_df.sort_values(by='timestamp', inplace=True)

cols_to_drop = ['timestamp', 'network_ips_dst', 'network_ips_src', 'network_ports_dst', 'network_ports_src']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_dataset.X.shape[1]

# CƠ CHẾ ATTENTION THEO THỜI GIAN (Temporal Attention)
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# ĐẠI TU MODEL CNN-BiLSTM
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) 
        
        # Đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        if x.size(-1) >=2:
            x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # CHUẨN HOÁ LAYER NORM THEO TỪNG TIME-STEP
        out = self.layer_norm(out)
        
        context_vector, attn_weights = self.attention(out)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out


In [10]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledDataset(Dataset):
    def __init__(self, df, max_samples_per_class=20000):
        # Trích xuất Features và Labels
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        
        print(f"Đang tính toán các mẫu hợp lệ và Undersampling...")
        
        # Mảng chứa tất cả index từ 0 đến len(df) - 1
        all_indices = np.arange(len(self.X))
        all_labels = self.y.numpy()
        
        self.valid_indices = []
        
        # Lặp qua từng class
        classes = np.unique(all_labels)
        rng = np.random.default_rng(seed=42)
        
        for c in classes:
            # Lấy tất cả index của các mẫu thuộc class c
            c_indices = all_indices[all_labels == c]
            count = len(c_indices)
            
            # Nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # Chọn ngẫu nhiên không hoàn lại
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} mẫu.")
            else:
                # Nếu ít hơn hoặc bằng ngưỡng thì giữ nguyên
                print(f"Class {c}: Giữ nguyên {count} mẫu.")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn khi train
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số mẫu sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # Lấy index thực sự đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # Lấy feature và label tại dòng (index) tương ứng
        sample_X = self.X[actual_idx].unsqueeze(0) 
        label_y = self.y[actual_idx]
        
        return sample_X, label_y

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 

BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, đưa mỗi sample riêng lẻ).")
train_dataset = UndersampledDataset(train_df, max_samples_per_class=MAX_SAMPLES)


print(f"Khởi tạo tập Val và Test (Không Undersampling, đưa mỗi sample riêng lẻ, giữ nguyên gốc)...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledDataset(valid_df, max_samples_per_class=10000000)
test_dataset = UndersampledDataset(test_df, max_samples_per_class=10000000)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, đưa mỗi sample riêng lẻ).
Đang tính toán các mẫu hợp lệ và Undersampling...
Class 0: Giảm từ 95760 xuống 20000 mẫu.
Class 1: Giữ nguyên 1307 mẫu.
Class 2: Giữ nguyên 12639 mẫu.
Class 3: Giữ nguyên 12894 mẫu.
Class 4: Giữ nguyên 5278 mẫu.
Class 5: Giữ nguyên 5643 mẫu.
Class 6: Giảm từ 23553 xuống 20000 mẫu.
Class 7: Giữ nguyên 1957 mẫu.
Tổng số mẫu sau khi lọc và Undersampling: 79718
Khởi tạo tập Val và Test (Không Undersampling, đưa mỗi sample riêng lẻ, giữ nguyên gốc)...
Đang tính toán các mẫu hợp lệ và Undersampling...
Class 0: Giữ nguyên 20520 mẫu.
Class 1: Giữ nguyên 280 mẫu.
Class 2: Giữ nguyên 2708 mẫu.
Class 3: Giữ nguyên 2763 mẫu.
Class 4: Giữ nguyên 1131 mẫu.
Class 5: Giữ nguyên 1209 mẫu.
Class 6: Giữ nguyên 5047 mẫu.
Class 7: Giữ nguyên 419 mẫu.
Tổng số mẫu sau khi lọc và Undersampling: 34077
Đang tính toán các mẫu hợp lệ và Undersampling...
Class 0: Giữ nguyên 20520 mẫu.
Class 1: Giữ nguyên 281 mẫu.
Class 2: Giữ nguyên 2709 mẫu.
Class 3:

In [12]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
# tạo folder để lưu model
import os
model_dir = "model_final"
os.makedirs(model_dir, exist_ok=True)

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=1).to(DEVICE)
# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val macro f1 không cải thiện sau 2 epochs (lưu ý đổi mode thành 'max')
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    # nếu là epoch chẵn thì không đánh giá trên tập val để tiết kiệm thời gian, chỉ đánh giá trên tập val ở các epoch lẻ

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Đưa val_macro_f1 vào scheduler (trước đó là avg_val_loss)
    scheduler.step(val_macro_f1)
    
    # Lưu model dựa trên Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_v2.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Macro F1 sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, zero_division=0, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 1.2134, Train Macro F1: 0.5872 | Val Loss: 0.9858, Val Macro F1: 0.5880


Epoch [2/60] - Train Loss: 0.9801, Train Macro F1: 0.7160 | Val Loss: 0.9492, Val Macro F1: 0.6208


Epoch [3/60] - Train Loss: 0.9395, Train Macro F1: 0.7354 | Val Loss: 0.9360, Val Macro F1: 0.6381


Epoch [4/60] - Train Loss: 0.9221, Train Macro F1: 0.7548 | Val Loss: 1.0431, Val Macro F1: 0.6225


...

Epoch [24/60] - Train Loss: 0.8304, Train Macro F1: 0.8372 | Val Loss: 0.9701, Val Macro F1: 0.6232


Epoch [25/60] - Train Loss: 0.8296, Train Macro F1: 0.8356 | Val Loss: 0.9490, Val Macro F1: 0.6601


Epoch [26/60] - Train Loss: 0.8275, Train Macro F1: 0.8401 | Val Loss: 0.9532, Val Macro F1: 0.6563


Epoch [27/60] - Train Loss: 0.8259, Train Macro F1: 0.8405 | Val Loss: 0.9642, Val Macro F1: 0.6343

[Early Stopping] Model không cải thiện Macro F1 sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.9283    0.9843    0.9555     20520
           1     0.0000    0.0000    0.0000       281
           2     0.5129    0.2566    0.3420      2709
           3     0.4961    0.5726    0.5316      2763
           4     0.6560    0.8004    0.7211      1132
           5     0.5354    0.2248    0.3166      1210
           6     0.8760    0.9527    0.9127      5048
           7     0.8889    0.8381    0.8627       420

    accuracy                         0.8454     34083
   macro avg     0.6117    0.5787    0.5803     34083
weighted avg     0.8214    0.8454    0.8265     34083



Undersampling + CNN + LSTM + Attention + Sliding Window

In [13]:
import pandas as pd
train_df = pd.read_csv(r"train.csv")
valid_df = pd.read_csv(r"val.csv")
test_df = pd.read_csv(r"test.csv")

train_df.sort_values(by='timestamp', inplace=True)
valid_df.sort_values(by='timestamp', inplace=True)
test_df.sort_values(by='timestamp', inplace=True)

cols_to_drop = ['timestamp', 'network_ips_dst', 'network_ips_src', 'network_ports_dst', 'network_ports_src']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [14]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # tạo mảng index cho step = 1 (dành riêng cho các class hiếm cần bảo toàn)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        # tạo mảng index theo step được truyền vào (cho các class còn lại)
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        
        # lặp qua từng class (Lấy từ step=1 để đảm bảo không sót class nào)
        classes = np.unique(window_labels_step1)
        rng = np.random.default_rng(seed=42)
        for c in classes:
            # ƯU TIÊN GIỮ NGUYÊN băm cửa sổ 1-1 cho Class 2 và 6
            if c in [100]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
                
            count = len(c_indices)
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                 
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [2, 6]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt tại index thực sự
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 1 
NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1) 
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, set Train Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giảm từ 95760 xuống 20000 cửa sổ.
Class 1: Giữ nguyên 1307 cửa sổ (step=1).
Class 2: Giữ nguyên 12639 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giữ nguyên 12894 cửa sổ (step=1).
Class 4: Giữ nguyên 5278 cửa sổ (step=1).
Class 5: Giữ nguyên 5643 cửa sổ (step=1).
Class 6: Giảm từ 23544 xuống 20000 cửa sổ.
Class 7: Giữ nguyên 1957 cửa sổ (step=1).
Tổng số cửa sổ sau khi lọc và Undersampling: 79718
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 20520 cửa sổ (step=1).
Class 1: Giữ nguyên 280 cửa sổ (step=1).
Class 2: Giữ nguyên 2708 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giữ nguyên 2763 cửa sổ (step=1).
Class 4: Giữ nguyên 1131 cửa sổ (step=1).
Class 5: Giữ nguyên 1209 cửa sổ (step=1).
Class 6: Giữ nguyên 5038 cửa sổ (

In [16]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        # lớp Linear để tính điểm số (score) cho từng time-step
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # đầu ra bi-lstm: (Batch, SeqLen, Hidden*2)
        
        # tính điểm chú tý cho mỗi timestap => (batch, seqlen, 1)
        scores = self.attention(lstm_outputs)
        
        # áp dung softmax để đưa sequence attention scores về dạng xác suất
        weights = torch.softmax(scores, dim=1)
        
        # nhân trọng số với output của lstm và tỉnh tổng (nhân chập)
        # (Batch, SeqLen, 1) * (Batch, SeqLen, Hidden*2) -> (Batch, Hidden*2)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        
        # trả lại context_vector và weights (để phục vụ cho việc trực quan hóa attention sau này)
        return context_vector, weights

# model CNN-BiLSTM với Cơ chế Attention
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # conv1d có padding=1 => giữ nguyên chiều dài chuỗi
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.dropout_cnn = nn.Dropout1d(0.2)

        # conv1d có padding=1 => giữ nguyên chiều dài chuỗi, tăng kênh lên 128
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        
        self.relu = nn.ReLU()
        
        # giảm 2 timesteps lại còn 1
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        # bi-lstm như cũ
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        

        # kích thước đầu vào của Attention = hidden_size * 2 (vì BiLSTM ghép 2 chiều ngược xuôi)
        self.attention = Attention(hidden_size * 2)
        
        self.dropout = nn.Dropout(0.5)
        
        # tầng phân loại cuối cùng
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # đầu vào (256,10, 314)
        
        # đảo trục cho CNN -> (256, 314, 10)
        x = x.permute(0, 2, 1) 
        
        # đi qua cnn block 1 => (256, 64, 10)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout_cnn(x)

        # đi qua cnn block 2 => (256, 128, 10)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        
        # giảm chiều dài chuỗi đi 2 lần => (256, 128, 5)
        x = self.pool(x)
        
        # đảo trục lại cho lstm (256, 5, 128)
        x = x.permute(0, 2, 1)
        
        # đi qua bi-lstm => (256, 5, 256)
        out, (hn, cn) = self.bilstm(x)
        
        # dùng attention để tính ra context vector: (256, 256)
        context_vector, attn_weights = self.attention(out)
        
        # đi qua lớp dense để ra dự đoán nhãn
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        # THAY ĐỔI DUY NHẤT Ở ĐÂY: in_channels = num_features (thay vì 1)
        # Conv1d sẽ trượt dọc theo chiều thời gian (Time_Steps) thay vì trượt dọc theo các features.
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        # tầng 2: giữ nguyên
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        # tầng 3: giữ nguyên
        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        # tầng BiLSTM: giữ nguyên
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        # tầng attention: giữ nguyên
        self.attention = Attention(128 * 2) 

        # các tầng dense: giữ nguyên
        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        # tầng phân loại cuối cùng: giữ nguyên
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Đầu vào x từ SlidingWindowDataset đang có shape: (Batch, Time_Steps, Features)
        
        # THAY ĐỔI Ở ĐÂY: Dùng permute thay vì unsqueeze
        # Chuyển vị để shape trở thành: (Batch, Features, Time_Steps)
        # Để Conv1d quét qua chiều Time_Steps với số kênh tương ứng với số Features
        x = x.permute(0, 2, 1) 

        # đi qua 3 tầng Conv1dCNN (Code đoạn này giữ nguyên 100%)
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        # permute để đưa qua BiLSTM: (Batch, Channels, Time_Steps_New) -> (Batch, Time_Steps_New, Channels)
        # Chiều Channels lúc này đang là 128 (khớp hoàn hảo với input_size=128 của BiLSTM)
        x = x.permute(0, 2, 1)

        # đi qua BiLSTM
        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        # đi qua attention để lấy context vector
        context_vector, _ = self.attention(out)

        # đi qua các lớp dense
        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        return out

In [18]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
# tạo folder để lưu model
import os
model_dir = "model_final"
os.makedirs(model_dir, exist_ok=True)

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val macro f1 không cải thiện sau 2 epochs (lưu ý đổi mode thành 'max')
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    # nếu là epoch chẵn thì không đánh giá trên tập val để tiết kiệm thời gian, chỉ đánh giá trên tập val ở các epoch lẻ

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Đưa val_macro_f1 vào scheduler (trước đó là avg_val_loss)
    scheduler.step(val_macro_f1)
    
    # Lưu model dựa trên Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_v2.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Macro F1 sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, zero_division=0, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 1.0214, Train Macro F1: 0.4989 | Val Loss: 0.7300, Val Macro F1: 0.5477


Epoch [2/60] - Train Loss: 0.6452, Train Macro F1: 0.8088 | Val Loss: 0.6954, Val Macro F1: 0.7071


Epoch [3/60] - Train Loss: 0.5892, Train Macro F1: 0.9181 | Val Loss: 0.7427, Val Macro F1: 0.7345


Epoch [4/60] - Train Loss: 0.5693, Train Macro F1: 0.9544 | Val Loss: 0.8249, Val Macro F1: 0.6850


...

Epoch [20/60] - Train Loss: 0.5103, Train Macro F1: 0.9821 | Val Loss: 0.8029, Val Macro F1: 0.7130


Epoch [21/60] - Train Loss: 0.5104, Train Macro F1: 0.9840 | Val Loss: 0.8019, Val Macro F1: 0.7102


Epoch [22/60] - Train Loss: 0.5102, Train Macro F1: 0.9837 | Val Loss: 0.8016, Val Macro F1: 0.7104


Epoch [23/60] - Train Loss: 0.5100, Train Macro F1: 0.9830 | Val Loss: 0.8031, Val Macro F1: 0.7100

[Early Stopping] Model không cải thiện Macro F1 sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.9842    0.9805    0.9823     20520
           1     0.0230    0.0178    0.0201       281
           2     0.7491    0.3747    0.4995      2709
           3     0.5907    0.8024    0.6805      2763
           4     0.9956    0.9947    0.9951      1132
           5     0.9787    0.3041    0.4641      1210
           6     0.8088    0.9996    0.8941      5039
           7     0.7260    0.9905    0.8379       420

    accuracy                         0.8893     34074
   macro avg     0.7320    0.6830    0.6717     34074
weighted avg     0.8968    0.8893    0.8787     34074



Undersampling + CNN + LSTM + Attention (attention + no sliding window) (cấu trúc giống paper gốc)

In [19]:
import pandas as pd
train_df = pd.read_csv(r"train.csv")
valid_df = pd.read_csv(r"val.csv")
test_df = pd.read_csv(r"test.csv")

train_df.sort_values(by='timestamp', inplace=True)
valid_df.sort_values(by='timestamp', inplace=True)
test_df.sort_values(by='timestamp', inplace=True)

cols_to_drop = ['timestamp', 'network_ips_dst', 'network_ips_src', 'network_ports_dst', 'network_ports_src']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_df.shape[1] - 1 # trừ đi cột label
NUM_CLASSES = len(train_df['label'].unique()) # số class
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # tính điểm chú ý
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        # Nhân trọng số và tính tổng
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# kiến trúc chính của paper
class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        # tầng 1: conv1d với 128 filters, kernel size = 5, padding = 2 để giũ nguyên chiều dài chuỗi
        # in_channels = 1 (Vì ta coi 314 features như 1 chuỗi dài 314 đơn vị) (đưa từng flow riêng lẻ vào)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        # tầng 2: conv1d với 256 filters, kernel size 5, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3)
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        # tầng 3: conv1d với 128 filters, kernel size 3, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3)
        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        # tầng BiLSTM: 128 units, Dropout(0.3)
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        # tầng attention
        self.attention = Attention(128 * 2) # *2 vì là Bidirectional

        # các tầng dense
        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        # tầng phân loại cuối cùng
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Đầu vào x từ StandardFlowDataset đang có shape: (Batch, Features)
        # đầu vào x có shape: (256, 314)
        
        # shape trở thành: (256, 1, 314)
        x = x.unsqueeze(1) 

        # đi qua 3 tầng Conv1dCNN
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        # permute để đưa qua BiLSTM
        x = x.permute(0, 2, 1)

        # đi qua BiLSTM
        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        # đi qua attention để lấy context vector
        context_vector, _ = self.attention(out)

        # đi qua các lớp dense
        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        # không dùng softmax
        return out

# khởi tạo mô hình và đẩy lên GPU
model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
print(model)

Paper_CNN_BiLSTM_Attention(
  (conv1): Conv1d(1, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop1): Dropout1d(p=0.3, inplace=False)
  (conv2): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop2): Dropout1d(p=0.3, inplace=False)
  (conv3): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop3): Dropout1d(p=0.3, inplace=False)
  (bilstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (drop_lstm): Dropout(p=0.3, inp

In [21]:
from torch.utils.data import Dataset
import numpy as np
class StandardFlowDataset(Dataset):
    def __init__(self, df, max_samples_per_class=None):
        self.X = []
        self.y = []
        
        # nhóm theo nhãn và áp dụng undersampling
        for class_name, group in df.groupby('label'):
            if max_samples_per_class and len(group) > max_samples_per_class:
                group = group.sample(n=max_samples_per_class, random_state=42)
            
            self.X.append(group.drop(columns=['label']).values)
            self.y.append(group['label'].values)
            
        self.X = np.vstack(self.X)
        self.y = np.concatenate(self.y)
        
        # xáo trộn dữ liệu sau khi đã áp dụng undersampling để đảm bảo tính ngẫu nhiên
        idx = np.random.permutation(len(self.X))
        self.X = torch.tensor(self.X[idx], dtype=torch.float32)
        self.y = torch.tensor(self.y[idx], dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# khởi tạo dataset với max_samples_per_class để giới hạn số lượng mẫu của mỗi class trong tập train, còn val và test thì giữ nguyên để đánh giá chính xác hơn
MAX_SAMPLES = 20000 
train_dataset = StandardFlowDataset(train_df, max_samples_per_class=MAX_SAMPLES)
val_dataset = StandardFlowDataset(valid_df)
test_dataset = StandardFlowDataset(test_df) 

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [22]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
# tạo folder để lưu model
import os
model_dir = "model_final"
os.makedirs(model_dir, exist_ok=True)

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val macro f1 không cải thiện sau 2 epochs (lưu ý đổi mode thành 'max')
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    # nếu là epoch chẵn thì không đánh giá trên tập val để tiết kiệm thời gian, chỉ đánh giá trên tập val ở các epoch lẻ

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Đưa val_macro_f1 vào scheduler (trước đó là avg_val_loss)
    scheduler.step(val_macro_f1)
    
    # Lưu model dựa trên Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_v2.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Macro F1 sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, zero_division=0, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 1.4411, Train Macro F1: 0.3780 | Val Loss: 0.9754, Val Macro F1: 0.4982


Epoch [2/60] - Train Loss: 1.1797, Train Macro F1: 0.5550 | Val Loss: 0.9605, Val Macro F1: 0.4936


Epoch [3/60] - Train Loss: 1.0903, Train Macro F1: 0.6191 | Val Loss: 0.9209, Val Macro F1: 0.6337


Epoch [4/60] - Train Loss: 1.0058, Train Macro F1: 0.6743 | Val Loss: 0.9226, Val Macro F1: 0.6811


...

Epoch [35/60] - Train Loss: 0.8538, Train Macro F1: 0.7922 | Val Loss: 0.8729, Val Macro F1: 0.6962


Epoch [36/60] - Train Loss: 0.8532, Train Macro F1: 0.7917 | Val Loss: 0.8686, Val Macro F1: 0.6982


Epoch [37/60] - Train Loss: 0.8538, Train Macro F1: 0.7913 | Val Loss: 0.8738, Val Macro F1: 0.6972


Epoch [38/60] - Train Loss: 0.8530, Train Macro F1: 0.7900 | Val Loss: 0.8732, Val Macro F1: 0.6966

[Early Stopping] Model không cải thiện Macro F1 sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.9283    0.9873    0.9569     20520
           1     0.0000    0.0000    0.0000       281
           2     0.5455    0.2078    0.3010      2709
           3     0.5738    0.7749    0.6594      2763
           4     0.6936    0.7420    0.7170      1132
           5     0.3503    0.1992    0.2540      1210
           6     0.9456    0.9604    0.9529      5048
           7     0.7232    0.8024    0.7607       420

    accuracy                         0.8576     34083
   macro avg     0.5951    0.5843    0.5752     34083
weighted avg     0.8332    0.8576    0.8368     34083



SMOTE

In [23]:
import pandas as pd

data_train = pd.read_csv("train.csv")

In [24]:
print(data_train['label'].value_counts())

label
0    95760
6    23553
3    12894
2    12639
5     5643
4     5278
7     1957
1     1307
Name: count, dtype: int64


In [25]:
import pandas as pd
train_df = pd.read_csv(r"train.csv")
valid_df = pd.read_csv(r"val.csv")
test_df = pd.read_csv(r"test.csv")

train_df.sort_values(by='timestamp', inplace=True)
valid_df.sort_values(by='timestamp', inplace=True)
test_df.sort_values(by='timestamp', inplace=True)

cols_to_drop = ['timestamp', 'network_ips_dst', 'network_ips_src', 'network_ports_dst', 'network_ports_src']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [26]:
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTETomek
y_train = train_df['label']
x_train = train_df.drop(columns=['label'])
TARGET_COUNT = 23000
counts = y_train.value_counts().to_dict()



under_strategy = {label: TARGET_COUNT for label, count in counts.items() if count > TARGET_COUNT}
over_strategy = {label: TARGET_COUNT for label, count in counts.items() if count < TARGET_COUNT}

under = RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)
smote_tomek = SMOTETomek(sampling_strategy=over_strategy, random_state=42, n_jobs=-1)

pipeline = Pipeline(steps=[('under', under), ('over', smote_tomek)])

print("Start to apply Hybrid Sampling (Under + Borderline SMOTE)...")
x_train_resampled, y_train_resampled = pipeline.fit_resample(x_train, y_train)

print(f"Number of samples after resampling: {len(y_train_resampled)}")
print(y_train_resampled.value_counts())
train_resampled_df = pd.DataFrame(x_train_resampled, columns=x_train.columns)
train_resampled_df['label'] = y_train_resampled

Start to apply Hybrid Sampling (Under + Borderline SMOTE)...
Number of samples after resampling: 181790
label
7    22976
1    22970
5    22925
4    22851
3    22814
2    22763
6    22302
0    22189
Name: count, dtype: int64


C:\Users\Admin\AppData\Local\Temp\ipykernel_12392\3707060278.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_resampled_df['label'] = y_train_resampled


In [27]:
from torch.utils.data import Dataset
import torch
import numpy as np
from torch.utils.data import DataLoader
class StandardFlowDataset(Dataset):
    def __init__(self, df, max_samples_per_class=None):
        self.X = []
        self.y = []
        
        for class_name, group in df.groupby('label'):
            if max_samples_per_class and len(group) > max_samples_per_class:
                group = group.sample(n=max_samples_per_class, random_state=42)
            
            self.X.append(group.drop(columns=['label']).values)
            self.y.append(group['label'].values)
            
        self.X = np.vstack(self.X)
        self.y = np.concatenate(self.y)
        
        idx = np.random.permutation(len(self.X))
        self.X = torch.tensor(self.X[idx], dtype=torch.float32)
        self.y = torch.tensor(self.y[idx], dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [28]:
BATCH_SIZE = 512
train_dataset = StandardFlowDataset(train_resampled_df)
val_dataset = StandardFlowDataset(valid_df)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_dataset = StandardFlowDataset(test_df)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_dataset.X.shape[1]
NUM_CLASSES = len(train_dataset.y.unique())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        self.attention = Attention(128 * 2) 

        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        
        x = x.unsqueeze(1) 

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        x = x.permute(0, 2, 1)

        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        context_vector, _ = self.attention(out)

        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        return out


In [30]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
import torch.optim as optim

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 1.4754, Train Macro F1: 0.4910 | Val Loss: 1.1970, Val Macro F1: 0.5160


Epoch [2/60] - Train Loss: 1.1642, Train Macro F1: 0.6485 | Val Loss: 1.1010, Val Macro F1: 0.5567


Epoch [3/60] - Train Loss: 1.0680, Train Macro F1: 0.7268 | Val Loss: 1.0753, Val Macro F1: 0.6172


Epoch [4/60] - Train Loss: 1.0034, Train Macro F1: 0.7660 | Val Loss: 1.0397, Val Macro F1: 0.6854


...

Epoch [36/60] - Train Loss: 0.8130, Train Macro F1: 0.8642 | Val Loss: 1.0004, Val Macro F1: 0.7017


Epoch [37/60] - Train Loss: 0.8130, Train Macro F1: 0.8643 | Val Loss: 0.9912, Val Macro F1: 0.7052


Epoch [38/60] - Train Loss: 0.8120, Train Macro F1: 0.8648 | Val Loss: 1.0000, Val Macro F1: 0.7004


Epoch [39/60] - Train Loss: 0.8116, Train Macro F1: 0.8651 | Val Loss: 1.0020, Val Macro F1: 0.7014

[Early Stopping] Model không cải thiện sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.9416    0.9790    0.9600     20520
           1     0.0581    0.0320    0.0413       281
           2     0.7438    0.2433    0.3666      2709
           3     0.6203    0.7959    0.6972      2763
           4     0.6946    0.8339    0.7579      1132
           5     0.6191    0.3157    0.4182      1210
           6     0.8531    0.9628    0.9046      5048
           7     0.7357    0.8548    0.7907       420

    accuracy                         0.8656     34083
   macro avg     0.6583    0.6272    0.6171     34083
weighted avg     0.8572    0.8656    0.8477     34083



In [31]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
import torch.optim as optim

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 1.4880, Train Macro F1: 0.4807 | Val Loss: 1.2203, Val Macro F1: 0.5346


Epoch [2/60] - Train Loss: 1.1561, Train Macro F1: 0.6621 | Val Loss: 1.1027, Val Macro F1: 0.5785


Epoch [3/60] - Train Loss: 1.0395, Train Macro F1: 0.7471 | Val Loss: 1.0644, Val Macro F1: 0.6423


Epoch [4/60] - Train Loss: 0.9942, Train Macro F1: 0.7678 | Val Loss: 1.0151, Val Macro F1: 0.6806


...

Epoch [25/60] - Train Loss: 0.8293, Train Macro F1: 0.8586 | Val Loss: 1.0016, Val Macro F1: 0.6971


Epoch [26/60] - Train Loss: 0.8296, Train Macro F1: 0.8584 | Val Loss: 1.0101, Val Macro F1: 0.6970


Epoch [27/60] - Train Loss: 0.8291, Train Macro F1: 0.8581 | Val Loss: 1.0049, Val Macro F1: 0.6978


Epoch [28/60] - Train Loss: 0.8290, Train Macro F1: 0.8581 | Val Loss: 1.0064, Val Macro F1: 0.6971

[Early Stopping] Model không cải thiện sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.9385    0.9760    0.9569     20520
           1     0.0469    0.0356    0.0405       281
           2     0.8840    0.2362    0.3729      2709
           3     0.5998    0.7546    0.6684      2763
           4     0.6287    0.7465    0.6826      1132
           5     0.5061    0.5479    0.5262      1210
           6     0.9190    0.9546    0.9365      5048
           7     0.8051    0.8262    0.8155       420

    accuracy                         0.8637     34083
   macro avg     0.6660    0.6347    0.6249     34083
weighted avg     0.8692    0.8637    0.8503     34083



In [ ]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
import torch.optim as optim

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 1.4571, Train Macro F1: 0.4990 | Val Loss: 1.1092, Val Macro F1: 0.5395


Epoch [2/60] - Train Loss: 1.1407, Train Macro F1: 0.6755 | Val Loss: 1.1519, Val Macro F1: 0.5768


Epoch [3/60] - Train Loss: 1.0373, Train Macro F1: 0.7495 | Val Loss: 1.0681, Val Macro F1: 0.6530


Epoch [4/60] - Train Loss: 0.9934, Train Macro F1: 0.7718 | Val Loss: 1.0318, Val Macro F1: 0.6767


...

Epoch [34/60] - Train Loss: 0.8178, Train Macro F1: 0.8624 | Val Loss: 0.9988, Val Macro F1: 0.7017


Epoch [35/60] - Train Loss: 0.8166, Train Macro F1: 0.8630 | Val Loss: 0.9919, Val Macro F1: 0.7033


Epoch [36/60] - Train Loss: 0.8163, Train Macro F1: 0.8631 | Val Loss: 0.9968, Val Macro F1: 0.7026

[Early Stopping] Model không cải thiện sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.9385    0.9823    0.9599     20520
           1     0.0644    0.0534    0.0584       281
           2     0.4107    0.1920    0.2616      2709
           3     0.5234    0.6156    0.5658      2763
           4     0.6712    0.8207    0.7385      1132
           5     0.5272    0.4488    0.4848      1210
           6     0.9527    0.9505    0.9516      5048
           7     0.8698    0.8429    0.8561       420

    accuracy                         0.8513     34083
   macro avg     0.6197    0.6132    0.6096     34083
weighted avg     0.8335    0.8513    0.8383     34083



: 